# 16.5 Typing Real Code — Generators, Async, Imports and Third-Party

**Prerequisites:** 16.1–16.4, 4.3 Generators, 6.3 Context Managers, 12.5 asyncio  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Typing **generators**: `Iterator[T]` vs `Generator[Y, S, R]`
- Typing **async**: `Coroutine`, `Awaitable`, `AsyncIterator` — and the forgotten `await`
- Typing **context managers**, both `@contextmanager` and the dunder form
- 🔴 **`cast` is an unchecked assertion** — a lie the checker believes
- `TYPE_CHECKING` — annotations that break import cycles
- `from __future__ import annotations`, and what it does
- Third-party libraries: stubs, `types-*` packages and `py.typed`
- 🔴 Why `--ignore-missing-imports` silences the error but **not** the `Any`

---

## From rules to code

**16.1–16.4** were the type system. This notebook is the parts of a real codebase where people
get stuck: generators, `async`, context managers, imports, and libraries someone else wrote.

None of it is new machinery — it is the same generics and protocols applied to the shapes you
actually meet.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py165_"))


def write(name, source, root=None):
    """Write a script into the scratch tree and return its name."""
    path = (root or WORK) / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return name


def mypy(name, source=None, *flags, root=None):
    """Type-check with mypy and return its report."""
    if source is not None:
        write(name, source, root)
    done = subprocess.run(
        [sys.executable, "-m", "mypy", name,
         "--cache-dir", str(WORK / ".mypy_cache"),
         "--no-color-output", "--no-error-summary", *flags],
        cwd=root or WORK, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    report = (done.stdout + done.stderr).strip() or "(mypy found nothing to report)"
    return (f"$ mypy {name} {' '.join(flags)}".rstrip() + "\n" + "-" * 68 + "\n"
            + report + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


def python(name, root=None):
    done = subprocess.run([sys.executable, name], cwd=root or WORK, capture_output=True,
                          text=True, encoding="utf-8", errors="replace", timeout=60)
    return (f"$ python {name}\n" + "-" * 68 + "\n"
            + (done.stdout + done.stderr).strip() + "\n" + "-" * 68
            + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

## Generators

**4.3** built generators. Their types confuse people because there are two annotations and the
simple one is almost always right.

| Annotation | Use when |
|---|---|
| `Iterator[T]` | 🔴 the normal case — you only `yield` values |
| `Iterable[T]` | the *parameter* type; accepts more (**16.3**) |
| `Generator[Yield, Send, Return]` | you use `.send()` or `return` a value |

```
def countdown(start: int) -> Iterator[int]:      you yield ints
def accumulate() -> Generator[int, int, str]:    yields int, receives int, returns str
                              ─┬─  ─┬─  ─┬─
                          yielded  sent  returned
```

In [ ]:
print(mypy("generators.py", r"""
    from collections.abc import Generator, Iterable, Iterator


    def countdown(start: int) -> Iterator[int]:
        while start > 0:
            yield start
            start -= 1


    def accumulate() -> Generator[int, int, str]:
        total = 0
        while total < 10:
            received = yield total
            total += received
        return "done"


    def chunk[T](source: Iterable[T], size: int) -> Iterator[list[T]]:
        batch: list[T] = []
        for item in source:
            batch.append(item)
            if len(batch) == size:
                yield batch
                batch = []
        if batch:
            yield batch


    reveal_type(countdown(3))
    reveal_type(next(countdown(3)))
    reveal_type(accumulate())
    reveal_type(chunk([1, 2, 3, 4], 2))

    for value in countdown(3):
        reveal_type(value)

    wrong: Iterator[str] = countdown(3)          # 🔴 yields int, not str
"""))

Note `chunk` — a **generic** generator (**16.3**), revealed as
`Iterator[list[int]]` when given a `list[int]`. That combination is extremely common in real
code: a helper that batches, filters or transforms whatever it is given.

## Async

**12.5** built `asyncio`. The one thing to internalise:

> 🔴 **An `async def` function returns a coroutine, not its result.** The result only appears
> after `await`.

That single fact explains almost every async type error.

In [ ]:
print(mypy("asynctypes.py", r"""
    import asyncio
    from collections.abc import AsyncIterator, Awaitable


    async def fetch(url: str) -> bytes:
        await asyncio.sleep(0)
        return b"payload"


    async def stream(urls: list[str]) -> AsyncIterator[bytes]:
        for url in urls:
            yield await fetch(url)


    def schedule(work: Awaitable[bytes]) -> None: ...


    reveal_type(fetch)


    async def main() -> None:
        reveal_type(await fetch("https://example.com"))
        async for chunk in stream(["a", "b"]):
            reveal_type(chunk)

        schedule(fetch("https://example.com"))       # an Awaitable - fine unawaited here

        data: bytes = fetch("https://example.com")   # 🔴 forgot the await
"""))

Read mypy's own notes: **"Maybe you forgot to use `await`?"**. The checker
recognises the mistake and names it.

| Annotation | Means |
|---|---|
| `Coroutine[Any, Any, T]` | what `async def ... -> T` actually returns |
| `Awaitable[T]` | 🔴 anything you can `await` — **prefer this in parameters** |
| `AsyncIterator[T]` | the return type of an `async def` that yields |
| `AsyncIterable[T]` | the parameter type for `async for` |

`Awaitable[T]` is to `Coroutine` what `Iterable` is to `Iterator` (**16.3**): the wider,
more accepting choice for a parameter — it also admits futures and tasks.

> **Mocking async code** needs `AsyncMock`, not `Mock` — see **15.5**.

## Context managers

**6.3** built both forms. Their types follow directly.

In [ ]:
print(mypy("contexts.py", r"""
    from collections.abc import Iterator
    from contextlib import contextmanager
    from types import TracebackType


    @contextmanager
    def spool(name: str) -> Iterator[list[str]]:
        # 🔴 Iterator[T], not ContextManager[T] - the decorator does the conversion
        buffer: list[str] = []
        yield buffer
        buffer.clear()


    class Connection:
        def __enter__(self) -> "Connection":
            return self

        def __exit__(
            self,
            exc_type: type[BaseException] | None,
            exc: BaseException | None,
            traceback: TracebackType | None,
        ) -> None:
            return None

        def query(self, sql: str) -> int:
            return 0


    with spool("jobs") as buffer:
        reveal_type(buffer)
        buffer.append("build-1")

    with Connection() as conn:
        reveal_type(conn)
        conn.query("SELECT 1")
"""))

🔴 The trap is the `@contextmanager` return type: you annotate what the
**generator** yields — `Iterator[list[str]]` — *not* `ContextManager[list[str]]`. The decorator
performs that conversion, and annotating the converted type is wrong.

For the class form, `__enter__` should return **`Self`** (**16.4**) in anything subclassable;
`"Connection"` is used above only to keep the example self-contained.

> `__exit__` returning `None` (or `False`) means "do not suppress the exception". Returning
> `bool` tells the checker it *might* suppress — which changes what code after the `with` block
> can assume.

## 🔴 `cast` — the lie the checker believes

`cast(T, value)` tells the checker "trust me, this is a `T`". It performs **no check** and
**does nothing at runtime** — it returns the object unchanged.

In [ ]:
print(mypy("cast_check.py", r"""
    from typing import cast

    raw: object = "not a number"

    number = cast(int, raw)          # no check, no conversion - just a claim
    reveal_type(number)
    print(number + 1)
"""))
print()
# The same file WITHOUT the reveal_type, so it can actually run (16.1: bare
# reveal_type is a mypy pseudo-function and a NameError at runtime).
print(python(write("cast_run.py", r"""
    from typing import cast

    raw: object = "not a number"

    number = cast(int, raw)
    print("cast returned:", repr(number), "of type", type(number).__name__)
    print(number + 1)
""")))

mypy revealed `int` and reported **nothing**. At runtime `cast` handed back
the same `str`, and `number + 1` raised `TypeError`.

That is `cast`'s entire nature: it is `# type: ignore` (**16.1**) with a nicer face, and it
fails the same way — silently, until it doesn't.

| Instead of `cast` | Prefer |
|---|---|
| "I know this is a `str`" | `assert isinstance(x, str)` — narrows **and** checks (**16.2**) |
| "this dict has the right shape" | a `TypedDict` and real validation at the boundary |
| "the library returns `Any`" | a stub, or a small typed wrapper you own |

🔴 Legitimate uses exist — a deserialiser that has already validated, or working around a
checker limitation — but every `cast` should carry a comment saying why it is safe.

## `TYPE_CHECKING` — annotations that need not exist at runtime

Two modules that refer to each other's types create an **import cycle**: `job.py` imports
`runner.py` for an annotation, `runner.py` imports `job.py` for one, and Python raises
`ImportError`.

`typing.TYPE_CHECKING` is `False` at runtime and `True` for the checker. Put the import inside
it and the cycle disappears — but then the name does not exist when the annotation is
*evaluated*, which is what `from __future__ import annotations` fixes by making all annotations
strings.

In [ ]:
PROJECT = WORK / "circular"

write("job.py", r"""
    from __future__ import annotations          # annotations become strings

    from typing import TYPE_CHECKING

    if TYPE_CHECKING:                           # False at runtime - no cycle
        from runner import Runner


    class Job:
        def __init__(self, job_id: str, runner: Runner) -> None:
            self.job_id = job_id
            self.runner = runner

        def start(self) -> None:
            self.runner.launch(self)
""", root=PROJECT)

write("runner.py", r"""
    from __future__ import annotations

    from job import Job


    class Runner:
        def launch(self, job: Job) -> None:
            print("launching", job.job_id)
""", root=PROJECT)

write("main.py", r"""
    from job import Job
    from runner import Runner

    Job("build-1", Runner()).start()
""", root=PROJECT)

print(mypy("main.py", None, root=PROJECT))
print()
print(python("main.py", root=PROJECT))

Type-checks cleanly **and** runs. The checker followed the annotation to
`Runner`; Python never imported it.

### `from __future__ import annotations`

It makes every annotation a **string**, evaluated only if something asks for it. Three
consequences:

| Effect | Detail |
|---|---|
| Forward references work | you can annotate with a class defined later, no quotes |
| `TYPE_CHECKING` imports work | the name need not exist at runtime |
| Modern syntax on older Pythons | `int | None` in a file running on 3.8 |
| 🔴 `get_type_hints()` may fail | it has to resolve the string, and the name may be gone |

That last row is the catch, and it is why libraries doing runtime introspection — `pydantic`,
`dataclasses` with `Annotated` (**16.2**) — sometimes break under it.

> **Version note.** PEP 649 changes this in **3.14**: annotations become *lazily evaluated*
> natively, so the `__future__` import stops being necessary. Code written today still uses it
> widely, and it remains harmless.

## Third-party libraries

Your code is typed. The library you call is someone else's problem. There are four cases:

| Case | What you get | What to do |
|---|---|---|
| Ships types (`py.typed` marker) | full checking | nothing — this is the good case |
| Stubs on PyPI (`types-requests`) | full checking | `pip install types-requests` |
| Bundled in `typeshed` (stdlib) | full checking | nothing |
| Untyped, no stubs | 🔴 an error, and `Any` | wrap it, or write a minimal stub |

In [ ]:
UNTYPED = r"""
    import nonexistent_vendor_lib               # not installed anywhere

    client = nonexistent_vendor_lib.Client()
    reveal_type(client)
    client.completely_made_up_method()          # nothing can be checked here
"""

print(mypy("untyped.py", UNTYPED))
print()
print(mypy("untyped.py", None, "--ignore-missing-imports"))

🔴 **Read the two runs together.** `--ignore-missing-imports` silenced the
error and exit code, but `reveal_type` still says **`Any`** — and
`client.completely_made_up_method()` is still unchecked in both.

> **Silencing the error does not restore the type information.** It only stops mypy telling you
> that it has none. Everything that touches that library is now `Any`, with the spreading
> behaviour from **16.1**.

The fix that actually works is to **wrap the untyped library** in a thin module you own and
type *that*. You then get checking everywhere except the wrapper, and — as **15.5** argued for
mocking — a single place to fake in tests.

In [ ]:
print(mypy("wrapper.py", r"""
    from typing import Any, Protocol

    import nonexistent_vendor_lib               # type: ignore[import-not-found]


    class Uploader(Protocol):
        def upload(self, key: str, payload: bytes) -> str: ...


    class VendorUploader:
        # The ONLY place that touches the untyped library. Everything past this
        # boundary is fully checked.
        def __init__(self) -> None:
            self._client: Any = nonexistent_vendor_lib.Client()

        def upload(self, key: str, payload: bytes) -> str:
            result = self._client.put(key, payload)
            return str(result)


    def archive(uploader: Uploader, key: str) -> str:
        return uploader.upload(key, b"payload")


    archive(VendorUploader(), "build-1.log")
    archive(VendorUploader(), 42)               # 🔴 checked, despite the untyped vendor
"""))

The vendor library is `Any` inside `VendorUploader` and **nowhere else**. The
mistake at the bottom was caught because everything past the boundary is typed against a
`Protocol` (**16.4**).

### Writing a stub

If you cannot wrap it, a `.pyi` file describes the parts you use. Stubs are annotations only —
every body is `...`:

```python
# stubs/vendor.pyi
class Client:
    def put(self, key: str, payload: bytes) -> str: ...

def connect(url: str, *, timeout: float = ...) -> Client: ...
```

Point mypy at it with `mypy_path = "stubs"`. You only need to describe what you actually call.

### Shipping types with your own package

Add an empty **`py.typed`** file to your package and include it in the distribution. Without it,
consumers get "no stubs" even though your code is fully annotated — the marker is how a checker
knows your inline types are meant to be used. Packaging detail lives in **17**.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Annotating a `@contextmanager` function as `ContextManager[T]`.** Annotate what the generator yields — `Iterator[T]` — and let the decorator convert it.
2. 🔴 **Forgetting `await` and assigning the coroutine.** mypy says *Maybe you forgot to use await?* — read it rather than fighting the annotation.
3. **Using `Generator[Y, S, R]` when you only yield.** `Iterator[T]` is shorter and right.
4. 🔴 **Reaching for `cast` to make an error go away.** It is an unchecked assertion; the demonstration here returns a `str` typed as `int` and crashes one line later.
5. **`--ignore-missing-imports` as a fix.** It hides the message, not the `Any`.
6. **Letting an untyped library leak everywhere.** Wrap it once and type the wrapper.
7. **A `TYPE_CHECKING` import without `from __future__ import annotations`** (before 3.14) — the annotation is evaluated at runtime and the name does not exist.
8. **Assuming `from __future__ import annotations` is free.** It breaks runtime introspection that resolves annotations, which some libraries rely on.
9. **Forgetting `py.typed`** when publishing a typed package — consumers see none of your annotations.
10. **Typing an async parameter as `Coroutine`** when `Awaitable` would also accept tasks and futures.

## Best Practices

- Use `Iterator[T]` for generators; reserve `Generator[Y, S, R]` for `.send()` or a return value.
- Take `Iterable[T]` and `Awaitable[T]` as parameters; return `Iterator[T]` and `Coroutine`.
- Return `Self` from `__enter__` in anything that might be subclassed.
- Treat every `cast` as needing a comment justifying why it is safe — prefer `assert isinstance` where you can.
- Put type-only imports behind `TYPE_CHECKING` to break cycles and cut import time.
- Wrap untyped third-party libraries at a single boundary and type that boundary with a `Protocol`.
- Install the `types-*` stub package before writing your own stubs.
- Ship `py.typed` with any package whose annotations you want consumers to see.

## Practice Exercises

Try these before moving on.

1. Type a generator that reads a file in chunks (**8.5**). Is it `Iterator[bytes]` or `Generator[bytes, None, None]`? Which reads better?
2. 🔴 Write an `async def` and assign its result without `await`. Read mypy's note, then fix it. Now type the parameter as `Awaitable` and pass a `Task`.
3. Convert a `@contextmanager` from **6.3** to the class form with `__enter__`/`__exit__`, typed with `Self`. Which version would you rather subclass?
4. 🔴 Use `cast` to claim a `str` is an `int`, then run it. Replace the `cast` with an `assert isinstance` and compare what the checker and the runtime each do.
5. Create two modules that import each other's types and reproduce the `ImportError`. Fix it with `TYPE_CHECKING`, and confirm mypy still resolves the annotation.
6. Take a library with no stubs, wrap it behind a `Protocol`, and prove the checker catches a wrong argument on your side of the boundary.
7. Write a three-line `.pyi` stub for one function of an untyped module and point `mypy_path` at it. What is the smallest useful stub?
8. **Interview question:** what does `cast` do at runtime, and why is it more dangerous than it looks?

---

## Version notes

| Version | Change |
|---|---|
| **3.14** | 🔴 **PEP 649** — annotations are evaluated lazily by the interpreter, so `from __future__ import annotations` is no longer needed for forward references |
| **3.13** | `warnings.deprecated` decorator, which checkers understand |
| **3.12** | `@override` (PEP 698) — the checker verifies a method really does override something |
| **3.11** | `Self` (**16.4**); `asyncio.TaskGroup` and `ExceptionGroup` typing |
| **3.10** | `TypeGuard`; `ParamSpec` (**16.3**) |
| **3.9** | `Annotated` in `typing`; `collections.abc` generics usable directly |
| **3.7** | `from __future__ import annotations` introduced (PEP 563) |

## Where next

| Notebook | Covers |
|---|---|
| **16.6** | adopting types in an existing codebase — config, strictness, CI |

## Related

- **4.3 Generators** — what `Iterator[T]` describes
- **6.3 Context Managers** — both forms typed here
- **12.5 asyncio** — coroutines, tasks and `async for`
- **16.1** — `Any` and how it spreads, which is what an untyped library gives you
- **16.4 Protocols** — the boundary type for a wrapper
- **15.5 Test Doubles** — the same "wrap what you do not own" argument, for mocking
- **17 Tooling, Packaging and Environments** — `py.typed`, stub packages and distribution